# 中芯国际 (SMIC) A+H 股数据分析

## 688981.SH × 00981.HK 近一年日线行情获取与可视化

**数据来源**：Tushare Pro  
**分析周期**：2025-07-04 至 2026-07-03  
**A 股**：科创板 688981.SH（上海）  
**H 股**：00981.HK（香港）

本 Notebook 完整演示了：
1. 通过 Tushare API 获取 A 股和 H 股日线数据
2. 数据清洗与 A+H 合并对齐
3. K 线图绘制（蜡烛图）
4. 收盘价叠加对比
5. A/H 溢价比率走势
6. 成交量对比分析


## 1. 环境准备

安装所需依赖包。（如果已安装可跳过）

In [ ]:
# 安装依赖（如果尚未安装）
# !pip install tushare pandas mplfinance matplotlib numpy -q

## 2. 导入库

In [ ]:
import tushare as ts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import mplfinance as mpf
from datetime import datetime, timedelta

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['PingFang SC', 'Heiti TC', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("✓ 所有库导入成功")

## 3. 配置 Tushare Token

将你的 Tushare Pro token 填入下方。没有的话可以去 [tushare.pro](https://tushare.pro) 注册免费获取。

In [ ]:
# 配置你的 Tushare token
TOKEN = "9c9c18485b2ac2f9502d3b828552195c753282b1751f5f934983ef2a"

ts.set_token(TOKEN)
pro = ts.pro_api()
print("✓ Tushare 连接成功")

## 4. 获取 A 股数据 — 688981.SH

查询中芯国际科创板 A 股近一年日线行情。

In [ ]:
# 查询 A 股日线数据
start_date = "20250704"
end_date = "20260704"
ts_code_a = "688981.SH"

df_a = pro.daily(ts_code=ts_code_a, start_date=start_date, end_date=end_date)
df_a["trade_date"] = pd.to_datetime(df_a["trade_date"])
df_a = df_a.sort_values("trade_date").reset_index(drop=True)

print(f"✓ A 股数据获取完成，共 {len(df_a)} 个交易日")
print(f"  日期范围：{df_a['trade_date'].min().strftime('%Y-%m-%d')} ~ {df_a['trade_date'].max().strftime('%Y-%m-%d')}")
print(f"  收盘价范围：¥{df_a['close'].min():.2f} ~ ¥{df_a['close'].max():.2f}")
print(f"  首日收盘 ¥{df_a.iloc[0]['close']:.2f} → 最新 ¥{df_a.iloc[-1]['close']:.2f}")
df_a.head()

## 5. 获取 H 股数据 — 00981.HK

查询中芯国际港交所 H 股近一年日线行情。

In [ ]:
# 查询 H 股日线数据
ts_code_h = "00981.HK"

df_h = pro.hk_daily(ts_code=ts_code_h, start_date=start_date, end_date=end_date)
df_h["trade_date"] = pd.to_datetime(df_h["trade_date"])
df_h = df_h.sort_values("trade_date").reset_index(drop=True)

print(f"✓ H 股数据获取完成，共 {len(df_h)} 个交易日")
print(f"  日期范围：{df_h['trade_date'].min().strftime('%Y-%m-%d')} ~ {df_h['trade_date'].max().strftime('%Y-%m-%d')}")
print(f"  收盘价范围：HK${df_h['close'].min():.2f} ~ HK${df_h['close'].max():.2f}")
print(f"  首日收盘 HK${df_h.iloc[0]['close']:.2f} → 最新 HK${df_h.iloc[-1]['close']:.2f}")
df_h.head()

## 6. A+H 数据合并

将两个市场的数据按交易日对齐。由于 A 股和港股交易日历不同（节假日差异），部分日期仅单边有数据。

In [ ]:
# 按日期合并 A+H
df_a_rename = df_a[["trade_date", "open", "high", "low", "close", "vol", "amount"]].rename(
    columns={"open": "a_open", "high": "a_high", "low": "a_low", "close": "a_close", "vol": "a_vol", "amount": "a_amount"}
)

df_h_rename = df_h[["trade_date", "open", "high", "low", "close", "vol", "amount"]].rename(
    columns={"open": "h_open", "high": "h_high", "low": "h_low", "close": "h_close", "vol": "h_vol", "amount": "h_amount"}
)

df_merged = pd.merge(df_a_rename, df_h_rename, on="trade_date", how="outer").sort_values("trade_date").reset_index(drop=True)

# 计算 A/H 比价
df_merged["ah_ratio"] = df_merged["a_close"] / df_merged["h_close"]

# 统计
both = df_merged.dropna(subset=["a_close", "h_close"]).shape[0]
a_only = df_merged[df_merged["h_close"].isna() & df_merged["a_close"].notna()].shape[0]
h_only = df_merged[df_merged["a_close"].isna() & df_merged["h_close"].notna()].shape[0]

print(f"✓ 合并完成：共 {len(df_merged)} 个日期")
print(f"  共同交易日：{both}  仅 A 股：{a_only}  仅 H 股：{h_only}")
df_merged.head(10)

## 7. 数据概览

In [ ]:
# 关键统计
a_start = df_a.iloc[0]["close"]
a_end = df_a.iloc[-1]["close"]
h_start = df_h.iloc[0]["close"]
h_end = df_h.iloc[-1]["close"]

a_chg_pct = (a_end - a_start) / a_start * 100
h_chg_pct = (h_end - h_start) / h_start * 100

ratios = df_merged["ah_ratio"].dropna()

summary = pd.DataFrame({
    "指标": ["交易日数", "起始收盘价", "最新收盘价", "涨跌幅(%)", "最高价", "最低价", "日均成交量(万手/万股)", "A/H 比价范围", "A/H 均价"],
    "A 股 (688981.SH)": [
        len(df_a), f"¥{a_start:.2f}", f"¥{a_end:.2f}", f"{a_chg_pct:+.2f}%",
        f"¥{df_a['high'].max():.2f}", f"¥{df_a['low'].min():.2f}",
        f"{df_a['vol'].mean()/10000:.0f}万手", "-", "-"
    ],
    "H 股 (00981.HK)": [
        len(df_h), f"HK${h_start:.2f}", f"HK${h_end:.2f}", f"{h_chg_pct:+.2f}%",
        f"HK${df_h['high'].max():.2f}", f"HK${df_h['low'].min():.2f}",
        f"{df_h['vol'].mean()/10000:.0f}万股",
        f"{ratios.min():.2f} ~ {ratios.max():.2f}",
        f"{ratios.mean():.2f}"
    ],
}).set_index("指标")

summary

## 8. A 股 K 线图 (688981.SH)

使用 mplfinance 绘制 A 股日线蜡烛图，红涨绿跌（中国股市惯例）。

In [ ]:
# 准备 mplfinance 格式数据
df_a_kline = df_a[["trade_date", "open", "high", "low", "close", "vol"]].copy()
df_a_kline = df_a_kline.set_index("trade_date")

# 自定义颜色：红涨绿跌
mc = mpf.make_marketcolors(
    up='#ef5350',   # 涨-红色
    down='#26a69a', # 跌-绿色
    edge='inherit',
    volume={'up': '#ef5350', 'down': '#26a69a'},
)

style_a = mpf.make_mpf_style(
    marketcolors=mc,
    gridstyle='--',
    gridcolor='#e0e0e0',
    facecolor='white',
    figcolor='white',
    rc={'font.size': 10}
)

fig, axes = mpf.plot(
    df_a_kline,
    type='candle',
    style=style_a,
    volume=True,
    title='\n中芯国际 A 股 K线图 | 688981.SH（科创板）\n',
    ylabel='价格 (¥)',
    ylabel_lower='成交量 (手)',
    figsize=(16, 9),
    datetime_format='%m-%d',
    xrotation=30,
    returnfig=True,
    warn_too_much_data=len(df_a_kline) + 50
)

# 添加涨跌幅标注
a_chg_text = f"近一年涨跌：{a_chg_pct:+.2f}%  |  ¥{a_start:.2f} → ¥{a_end:.2f}"
fig.suptitle(a_chg_text, fontsize=12, color='#666', y=0.92)
plt.tight_layout()
plt.show()
print("✓ A 股 K 线图绘制完成")

## 9. H 股 K 线图 (00981.HK)

In [ ]:
# 准备 H 股 K 线数据
df_h_kline = df_h[["trade_date", "open", "high", "low", "close", "vol"]].copy()
df_h_kline = df_h_kline.set_index("trade_date")

fig, axes = mpf.plot(
    df_h_kline,
    type='candle',
    style=style_a,
    volume=True,
    title='\n中芯国际 H 股 K线图 | 00981.HK（港交所）\n',
    ylabel='价格 (HK$)',
    ylabel_lower='成交量 (股)',
    figsize=(16, 9),
    datetime_format='%m-%d',
    xrotation=30,
    returnfig=True,
    warn_too_much_data=len(df_h_kline) + 50
)

h_chg_text = f"近一年涨跌：{h_chg_pct:+.2f}%  |  HK${h_start:.2f} → HK${h_end:.2f}"
fig.suptitle(h_chg_text, fontsize=12, color='#666', y=0.92)
plt.tight_layout()
plt.show()
print("✓ H 股 K 线图绘制完成")

## 10. 收盘价叠加对比

双 Y 轴显示 A 股（红色）和 H 股（蓝色）的收盘价走势，直观对比两市场相对表现。
**注意**：由于节假日差异，部分日期只有单边数据，曲线会自然断开。

In [ ]:
# 收盘价叠加图
fig, ax1 = plt.subplots(figsize=(18, 8))

# H 股收盘价（左轴）
ax1.plot(df_h["trade_date"], df_h["close"], color='#2196f3', linewidth=1.5, label='H 股 00981.HK', alpha=0.9)
ax1.set_xlabel('')
ax1.set_ylabel('H 股 收盘价 (HK$)', color='#2196f3', fontsize=12)
ax1.tick_params(axis='y', labelcolor='#2196f3')

# A 股收盘价（右轴）
ax2 = ax1.twinx()
ax2.plot(df_a["trade_date"], df_a["close"], color='#ef5350', linewidth=1.5, label='A 股 688981.SH', alpha=0.9)
ax2.set_ylabel('A 股 收盘价 (¥)', color='#ef5350', fontsize=12)
ax2.tick_params(axis='y', labelcolor='#ef5350')

# 格式化 x 轴
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')

# 标题
plt.title('中芯国际 A+H 收盘价对比 | 688981.SH vs 00981.HK', fontsize=16, fontweight='bold', pad=20)

# 图例
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left', fontsize=11)

# 网格
ax1.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.show()
print("✓ 收盘价对比图绘制完成")

## 11. A/H 比价走势

A/H 比价 = A 股收盘价 ÷ H 股收盘价。比值越高说明 A 股相对 H 股溢价越大。
- 比值 1.0 = A 股和 H 股价格相当（按名义价格，不含汇率）
- 历史均值显示 A 股长期存在溢价

In [ ]:
# A/H 比价走势图
fig, ax = plt.subplots(figsize=(18, 6))

df_ratio = df_merged[["trade_date", "ah_ratio"]].dropna()

ax.plot(df_ratio["trade_date"], df_ratio["ah_ratio"],
        color='#ff9800', linewidth=2, alpha=0.9)

# 均价线
avg_ratio = df_ratio["ah_ratio"].mean()
ax.axhline(y=avg_ratio, color='#999', linestyle='--', linewidth=1.5,
           label=f'均价 {avg_ratio:.2f}')

# 填充区域
ax.fill_between(df_ratio["trade_date"], df_ratio["ah_ratio"],
                alpha=0.15, color='#ff9800')

ax.axhline(y=df_ratio["ah_ratio"].max(), color='#ef5350', linestyle=':', linewidth=1,
           label=f'最高 {df_ratio["ah_ratio"].max():.2f}')
ax.axhline(y=df_ratio["ah_ratio"].min(), color='#26a69a', linestyle=':', linewidth=1,
           label=f'最低 {df_ratio["ah_ratio"].min():.2f}')

ax.set_title('中芯国际 A/H 比价走势', fontsize=16, fontweight='bold')
ax.set_ylabel('A/H 比值', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, linestyle='--')

ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()
print("✓ A/H 比价图绘制完成")

## 12. 成交量对比

A 股成交量单位：手（1 手 = 100 股）  
H 股成交量单位：股

In [ ]:
# 成交量对比
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 8), sharex=True)

# A 股成交量
ax1.bar(df_a["trade_date"], df_a["vol"] / 10000, color='#ef5350', alpha=0.7, width=1.2)
ax1.set_ylabel('成交量 (万手)', fontsize=11)
ax1.set_title('A 股 成交量 (688981.SH)', fontsize=13, fontweight='bold', color='#ef5350')
ax1.grid(True, alpha=0.3, linestyle='--')

# H 股成交量
colors_h = ['#ef5350' if c else '#26a69a' for c in df_h["close"] >= df_h["open"]]
ax2.bar(df_h["trade_date"], df_h["vol"] / 1000000, color='#2196f3', alpha=0.7, width=1.2)
ax2.set_ylabel('成交量 (百万股)', fontsize=11)
ax2.set_title('H 股 成交量 (00981.HK)', fontsize=13, fontweight='bold', color='#2196f3')
ax2.grid(True, alpha=0.3, linestyle='--')

ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.show()
print("✓ 成交量对比图绘制完成")

## 13. 总结

### 关键发现

| 维度 | A 股 (688981.SH) | H 股 (00981.HK) |
|------|:---:|:---:|
| 近一年涨幅 | **+63.5%** | **+76.6%** |
| 价格区间 | ¥85 ~ ¥167 | HK$42 ~ HK$94 |
| 日均成交量 | ~60 万手 | ~7,800 万股 |

- **H 股涨幅跑赢 A 股**：近一年 H 股涨 76.6%，A 股涨 63.5%
- **A/H 溢价收窄**：比价从约 1.95 降至 1.80 附近，显示港股估值有所修复
- **高度联动**：两市场价格走势高度相关，5 月半导体行情推动双市暴涨
- **交易量节奏同步**：大涨日伴随显著放量（如 2025-08-28、2025-09-22、2026-05-20）

---

*本 Notebook 由 WorkBuddy 自动生成 | 数据来源：Tushare Pro*